# **Pathway Enrichment and Construction of Layered Tripartite Networks (LTNs) Involving MicroRNAs, Messenger RNAs, and Pathways**

This notebook also produces the AT_MN family of artifacts 

# Import Libraries and Configurations

In [1]:
import ast
import json
import os
import sys

import numpy as np
import pandas as pd
import requests

# Add project root to Python's path
sys.path.append(os.path.abspath(os.path.join('..')))

from config import (
    GP_API_ENDPOINTS,
    GP_DATA_DIRS,
    GP_FILE_SUFFIX,
    NETWORK_DATA_DIRS,
    NETWORK_FILES,
    PATHWAY_ENRICHMENT_SETUP,
)

# Pathway Enrichment

## Functions

### g:Profiler Pathway Enrichment

In [2]:

def gp_pathway_enrichment(genes_df):
    """
    Perform functional pathway enrichment analysis using the g:Profiler API.

    This function retrieves a list of genes/messenger RNAs from a DataFrame,
    submits them to the g:Profiler GOSt API for functional enrichment
    analysis, and processes the response into a DataFrame containing the
    enrichment results. The analysis parameters and API endpoint are
    obtained from global configuration dictionaries.

    Parameters
    ----------
    genes_df : pandas.DataFrame
        DataFrame containing the genes/messenger RNAs to be analyzed.
        Must include an `id` column with gene identifiers.

    Returns
    -------
    pandas.DataFrame
        DataFrame containing the functional enrichment results returned
        by g:Profiler, with an additional column `genes` listing the
        input genes associated with each enrichment result.

    Notes
    -----
    - The following global configuration variables are used:
        * `PATHWAY_ENRICHMENT_SETUP` : Parameters for functional
          enrichment analysis, including organism, annotation sources,
          significance threshold, and multiple testing correction method.
        * `GP_API_ENDPOINTS` : API endpoints for g:Profiler services.
    - The function uses the GOSt API to identify enriched functional
      terms and pathways associated with the input genes.
    - The `genes` column is derived from the `intersections` field
      returned by g:Profiler, preserving the input gene identifiers.
    - An HTTP error raises an exception if the API request fails.

    Examples
    --------
    >>> enrichment_df = gp_pathway_enrichment(genes_df)
    >>> enrichment_df[['source', 'native', 'name', 'p_value', 'genes']].head()
    """
    # List all genes/messenger RNAs in the network
    query = genes_df['id'].to_list()

    # Define the parameters for functional enrichment analysis
    parameters = {
        'organism': PATHWAY_ENRICHMENT_SETUP['organism'],
        'query': query,
        'sources': PATHWAY_ENRICHMENT_SETUP['sources'],
        'user_threshold': PATHWAY_ENRICHMENT_SETUP['user-threshold'],
        'all_results': PATHWAY_ENRICHMENT_SETUP['all-results'],
        'significance_threshold_method': \
            PATHWAY_ENRICHMENT_SETUP['significance-threshold-method'],
    }

    # Request analysis for the g:Profile API
    response = requests.post(
        url=GP_API_ENDPOINTS['GOSt'],
        json=parameters,
    )
    response.raise_for_status()

    # Convert the response content to a JSON
    json_response = json.loads(response.content.decode('utf-8'))

    # Transform the JSON response result into a DataFrame
    enrichment_df = pd.json_normalize(json_response['result'])

    # Recover the enriched genes/messenger RNAs
    enrichment_df['genes'] = enrichment_df['intersections'] \
        .apply(
            lambda col: [
                genes for genes, sources in zip(query, col) if sources
            ]
        )

    return enrichment_df

### Perform and Flag Pathway Enrichment

In [3]:

def perform_and_flag_pathway_enrichment(group):
    """
    Perform and flag functional pathway enrichment results of interest.

    This function loads the nodes of a microRNA-mRNA interaction network
    for a given group, extracts the messenger RNAs, and performs functional
    enrichment analysis using g:Profiler. The enrichment results are
    stored in an external CSV file, and pathways meeting a predefined
    significance threshold are flagged and stored in a processed CSV
    file. The threshold, file paths, and filenames are obtained from
    global configuration dictionaries.

    Parameters
    ----------
    group : str
        The name of the group for which functional pathway enrichment
        analysis will be performed. The normalized group name must
        correspond to a key in `NETWORK_DATA_DIRS['processed']`.

    Returns
    -------
    pandas.DataFrame
        DataFrame containing the functional enrichment results for the
        group, with an additional column `is_pathway_of_interest`
        indicating flagged pathways:
        - 1 : Pathway meets the significance threshold.
        - 0 : Pathway does not meet the significance threshold.

    Notes
    -----
    - The following global configuration variables are used:
        * `NETWORK_DATA_DIRS` : Directory paths for processed network data.
        * `NETWORK_FILES` : Filenames for specific network data files.
        * `GP_DATA_DIRS` : Directory paths for external and processed
          enrichment data.
        * `GP_FILE_SUFFIX` : Filename suffix for enrichment results.
        * `PATHWAY_ENRICHMENT_SETUP` : Parameters for functional
          enrichment analysis, including the significance threshold.
    - The group name is normalized by converting it to lowercase and
      replacing spaces with hyphens.
    - Only nodes classified as `Messenger RNA` are submitted for
      functional enrichment analysis.
    - The function uses `gp_pathway_enrichment()` to retrieve the
      enrichment results from g:Profiler.
    - Enrichment results are flagged when their `p_value` is less than
      or equal to the configured significance threshold.
    - The function overwrites the corresponding CSV files in the
      external and processed directories.

    Examples
    --------
    >>> enrichment_df = perform_and_flag_pathway_enrichment('HER2-enriched')
    >>> enrichment_df[['name', 'p_value', 'is_pathway_of_interest']].head()
    """
    # Define the path to the directory related to the group
    group_dir = (group.lower()).replace(' ', '-')
    dir_path = NETWORK_DATA_DIRS['processed'][group_dir]

    # Create a node DataFrame for the microRNA-mRNA interaction network
    file_name = NETWORK_FILES['interaction-nodes']
    interaction_nodes_df = pd.read_csv(os.path.join(dir_path, file_name))
    
    # Pathway enrichment analysis of genes/messenger RNAs
    genes_df = interaction_nodes_df.query('type == "Messenger RNA"')
    enrichment_df = gp_pathway_enrichment(genes_df)

    # Store the group pathway enrichment DataFrame in a CSV file
    file_name = group_dir + GP_FILE_SUFFIX
    dir_path = GP_DATA_DIRS['external']
    enrichment_df.to_csv(
        os.path.join(dir_path, file_name), index=False
    )

    # Flag the enriched pathways of interest
    user_threshold = PATHWAY_ENRICHMENT_SETUP['user-threshold']
    enrichment_df['is_pathway_of_interest'] = np.where(
        enrichment_df['p_value'] <= user_threshold, 1, 0
    )

    # Store the flagged group pathway enrichment DataFrame in a CSV file
    dir_path = GP_DATA_DIRS['processed']
    enrichment_df.to_csv(
        os.path.join(dir_path, file_name), index=False
    )
    
    return enrichment_df

## Data Versions Endpoint

In [4]:
# Request the API version and status to the endpoint
response = requests.get(
    url=GP_API_ENDPOINTS['data_versions'],
    json={'organism': PATHWAY_ENRICHMENT_SETUP['organism']}
)

# Print the response content
try:
    parsed = json.loads(response.content.decode('utf-8'))
    print(json.dumps(parsed, indent=3))
except json.JSONDecodeError:
    print('Response is not valid JSON:')
    print(response.content.decode('utf-8'))

{
   "biomart": "Ensembl",
   "biomart_version": "114",
   "display_name": "Human",
   "genebuild": "GRCh38.p14",
   "gprofiler_version": "e114_eg62_p19_27110d83",
   "organism": "hsapiens",
   "sources": {
      "CORUM": {
         "name": "CORUM protein complexes",
         "version": "28.11.2022 Corum 4.1"
      },
      "GO:BP": {
         "name": "biological process",
         "version": "annotations: BioMart\nclasses: releases/2026-01-23"
      },
      "GO:CC": {
         "name": "cellular component",
         "version": "annotations: BioMart\nclasses: releases/2026-01-23"
      },
      "GO:MF": {
         "name": "molecular function",
         "version": "annotations: BioMart\nclasses: releases/2026-01-23"
      },
      "HP": {
         "name": "Human Phenotype Ontology",
         "version": "annotations: 03.2026\nclasses: None"
      },
      "HPA": {
         "name": "Human Protein Atlas",
         "version": "annotations: HPA website: 25-11-06\nclasses: script: 26-01-20"
 

## g:GOSt Endpoint

### Basal-like

In [5]:
# Perform pathway enrichment for the gene set related to the group
group = 'Basal-like'
enrichment_df = perform_and_flag_pathway_enrichment(group)

In [6]:
# Print the group's pathway enrichment DataFrame
enrichment_df

,description,effective_domain_size,goshv,intersection_size,intersections,name,native,p_value,parents,precision,query,query_size,recall,significant,source,term_size,source_order,group_id,genes,is_pathway_of_interest
0,Phase 0 - rapid depolarisation,11056,41125,5,"[[], [], [], [], [], [], [], [], [], [], [], [...",Phase 0 - rapid depolarisation,REAC:R-HSA-5576892,0.038079,[REAC:R-HSA-5576891],0.034722,query_1,144,0.156250,True,REAC,32,1801,1,"[HAS2, ZNF423, NMT1, TIA1, SYT11]",1
1,Nuclear Receptor transcription pathway,11056,41005,5,"[[], [], [], [], [], [], [], [], [], [], [], [...",Nuclear Receptor transcription pathway,REAC:R-HSA-383280,0.064488,[REAC:R-HSA-212436],0.034722,query_1,144,0.121951,True,REAC,41,1681,1,"[RXRG, TP53INP2, ARHGEF3, SMARCD1, ELFN2]",1
2,Calmodulin induced events,11056,39642,4,"[[], [], [], [], [], [], [], [], [], [], [], [...",Calmodulin induced events,REAC:R-HSA-111933,0.140741,[REAC:R-HSA-111997],0.027778,query_1,144,0.117647,False,REAC,34,318,1,"[HAS2, PRDM16, CHST1, SSTR1]",0
3,Ca-dependent events,11056,39635,4,"[[], [], [], [], [], [], [], [], [], [], [], [...",Ca-dependent events,REAC:R-HSA-111996,0.140741,[REAC:R-HSA-112043],0.027778,query_1,144,0.111111,False,REAC,36,311,1,"[HAS2, PRDM16, CHST1, SSTR1]",0
4,Activation of NMDA receptors and postsynaptic ...,11056,39419,6,"[[], [], [], [], [], [], [], [], [], [], [], [...",Activation of NMDA receptors and postsynaptic ...,REAC:R-HSA-442755,0.140741,[REAC:R-HSA-112314],0.041667,query_1,144,0.068182,False,REAC,88,95,1,"[RASGRF2, PPM1F, HAS2, PRDM16, ANKRD44, CHST1]",0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
713,DNA Repair,11056,39788,1,"[[], [], [], [], [], [], [], [], [], [], [], [...",DNA Repair,REAC:R-HSA-73894,0.991605,[REAC:0000000],0.006944,query_1,144,0.003040,False,REAC,329,464,1,[COPS7B],0
714,Class A/1 (Rhodopsin-like receptors),11056,39705,1,"[[], [], [], [], [], [], [], [], [], [], [], [...",Class A/1 (Rhodopsin-like receptors),REAC:R-HSA-373076,0.991605,[REAC:R-HSA-500792],0.006944,query_1,144,0.003049,False,REAC,328,381,1,[GDI2],0
715,SARS-CoV Infections,11056,41522,1,"[[], [], [], [], [], [], [], [], [], [], [], [...",SARS-CoV Infections,REAC:R-HSA-9679506,1.000000,[REAC:R-HSA-9824446],0.006944,query_1,144,0.002179,False,REAC,459,2198,1,[IL6ST],0
716,Metabolism,11056,40803,15,"[[], [], [], [], [], [], [], [], [], [], [], [...",Metabolism,REAC:R-HSA-1430728,1.000000,[REAC:0000000],0.104167,query_1,144,0.007000,False,REAC,2143,1479,1,"[SLC5A6, ICOS, ADAM19, YPEL5, EIF2S2, SMARCD1,...",0


### HER2-enriched

In [7]:
# Perform pathway enrichment for the gene set related to the group
group = 'HER2-enriched'
enrichment_df = perform_and_flag_pathway_enrichment(group)

In [8]:
# Print the group's pathway enrichment DataFrame
enrichment_df

,description,effective_domain_size,goshv,intersection_size,intersections,name,native,p_value,parents,precision,query,query_size,recall,significant,source,term_size,source_order,group_id,genes,is_pathway_of_interest
0,Developmental Biology,11056,40088,6,"[[], [REAC], [], [], [], [REAC], [REAC], [], [...",Developmental Biology,REAC:R-HSA-1266738,0.075917,[REAC:0000000],0.6,query_1,10,0.004304,True,REAC,1394,764,1,"[MEF2D, SMARCC1, SALL1, PRKAR2A, VAV3, SNAI1]",1
1,ADORA2B mediated anti-inflammatory cytokines p...,11056,39334,1,"[[], [], [], [], [], [], [], [], [REAC], [], [...",ADORA2B mediated anti-inflammatory cytokines p...,REAC:R-HSA-9660821,0.124276,[REAC:R-HSA-9662851],0.1,query_1,10,0.023810,False,REAC,42,10,1,[PRKAR2A],0
2,Ca-dependent events,11056,39635,1,"[[], [], [], [], [], [], [], [], [REAC], [], [...",Ca-dependent events,REAC:R-HSA-111996,0.124276,[REAC:R-HSA-112043],0.1,query_1,10,0.027778,False,REAC,36,311,1,[PRKAR2A],0
3,Azathioprine ADME,11056,39531,1,"[[], [], [], [], [], [], [], [], [], [REAC], [...",Azathioprine ADME,REAC:R-HSA-9748787,0.124276,[REAC:R-HSA-9748784],0.1,query_1,10,0.043478,False,REAC,23,207,1,[VAV3],0
4,Calmodulin induced events,11056,39642,1,"[[], [], [], [], [], [], [], [], [REAC], [], [...",Calmodulin induced events,REAC:R-HSA-111933,0.124276,[REAC:R-HSA-111997],0.1,query_1,10,0.029412,False,REAC,34,318,1,[PRKAR2A],0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
138,RNA Polymerase II Transcription,11056,41287,1,"[[], [], [], [], [], [REAC], [], [], [], [], [...",RNA Polymerase II Transcription,REAC:R-HSA-73857,0.749778,[REAC:R-HSA-74160],0.1,query_1,10,0.000740,False,REAC,1352,1963,1,[SMARCC1],0
139,Gene expression (Transcription),11056,40415,1,"[[], [], [], [], [], [REAC], [], [], [], [], [...",Gene expression (Transcription),REAC:R-HSA-74160,0.793425,[REAC:0000000],0.1,query_1,10,0.000650,False,REAC,1539,1091,1,[SMARCC1],0
140,Metabolism of proteins,11056,40821,1,"[[REAC], [], [], [], [], [], [], [], [], [], [...",Metabolism of proteins,REAC:R-HSA-392499,0.882711,[REAC:0000000],0.1,query_1,10,0.000484,False,REAC,2066,1497,1,[IGF2],0
141,Immune System,11056,40543,1,"[[], [], [], [], [], [], [], [], [], [REAC], [...",Immune System,REAC:R-HSA-168256,0.882711,[REAC:0000000],0.1,query_1,10,0.000479,False,REAC,2086,1219,1,[VAV3],0


### Luminal A

In [9]:
# Perform pathway enrichment for the gene set related to the group
group = 'Luminal A'
enrichment_df = perform_and_flag_pathway_enrichment(group)

In [10]:
# Print the group's pathway enrichment DataFrame
enrichment_df

,description,effective_domain_size,goshv,intersection_size,intersections,name,native,p_value,parents,precision,query,query_size,recall,significant,source,term_size,source_order,group_id,genes,is_pathway_of_interest
0,Extracellular matrix organization,11056,40254,14,"[[], [], [], [], [], [], [REAC], [], [], [], [...",Extracellular matrix organization,REAC:R-HSA-1474244,0.003062,[REAC:0000000],0.114754,query_1,122,0.046980,True,REAC,298,930,1,"[SERPINH1, LAMA2, ADGRL3, CELF2, PPP2R2B, PRKG...",1
1,Signaling by Receptor Tyrosine Kinases,11056,41723,18,"[[], [], [], [REAC], [], [], [], [], [REAC], [...",Signaling by Receptor Tyrosine Kinases,REAC:R-HSA-9006934,0.004169,[REAC:R-HSA-162582],0.147541,query_1,122,0.034884,True,REAC,516,2399,1,"[PCSK5, KIT, LAMA2, PDGFRA, AR, PPP2R2B, PRKG1...",1
2,Signal Transduction,11056,41633,49,"[[REAC], [], [], [REAC], [], [], [], [], [REAC...",Signal Transduction,REAC:R-HSA-162582,0.004345,[REAC:0000000],0.401639,query_1,122,0.019066,True,REAC,2570,2309,1,"[WWTR1, PCSK5, KIT, SOX17, WIPF1, LAMA2, ARL4C...",1
3,Collagen formation,11056,39727,7,"[[], [], [], [], [], [], [REAC], [], [], [], [...",Collagen formation,REAC:R-HSA-1474290,0.006863,[REAC:R-HSA-1474244],0.057377,query_1,122,0.078652,True,REAC,89,403,1,"[SERPINH1, ADGRL3, CELF2, PPP2R2B, PRKG1, LRMD...",1
4,Signaling by TGFB family members,11056,41731,9,"[[REAC], [], [], [], [], [], [], [], [], [], [...",Signaling by TGFB family members,REAC:R-HSA-9006936,0.006863,[REAC:R-HSA-162582],0.073770,query_1,122,0.057692,True,REAC,156,2407,1,"[WWTR1, CRTAP, IGSF11, MYLK, MSRB3, ITGA9, LMO...",1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
617,M Phase,11056,40722,1,"[[], [], [], [], [], [], [], [], [], [], [], [...",M Phase,REAC:R-HSA-68886,0.994762,[REAC:R-HSA-69278],0.008197,query_1,122,0.002457,False,REAC,407,1398,1,[TBX3],0
618,Viral Infection Pathways,11056,42066,4,"[[], [], [], [], [], [], [], [], [], [], [], [...",Viral Infection Pathways,REAC:R-HSA-9824446,0.994762,[REAC:R-HSA-5663205],0.032787,query_1,122,0.004582,False,REAC,873,2742,1,"[AR, LRRC28, VCL, B3GALT1]",0
619,GPCR ligand binding,11056,40390,1,"[[], [], [], [], [], [], [], [], [], [], [], [...",GPCR ligand binding,REAC:R-HSA-500792,0.997767,[REAC:R-HSA-372790],0.008197,query_1,122,0.002174,False,REAC,460,1066,1,[CBX7],0
620,Sensory Perception,11056,41617,1,"[[], [], [], [], [], [], [], [], [], [], [], [...",Sensory Perception,REAC:R-HSA-9709957,1.000000,[REAC:0000000],0.008197,query_1,122,0.001736,False,REAC,576,2293,1,[C2CD2],0


### Luminal B

In [11]:
# Perform pathway enrichment for the gene set related to the group
group = 'Luminal B'
enrichment_df = perform_and_flag_pathway_enrichment(group)

In [12]:
# Print the group's pathway enrichment DataFrame
enrichment_df

,description,effective_domain_size,goshv,intersection_size,intersections,name,native,p_value,parents,precision,query,query_size,recall,significant,source,term_size,source_order,group_id,genes,is_pathway_of_interest
0,Signaling by Receptor Tyrosine Kinases,11056,41723,36,"[[], [], [], [], [REAC], [], [], [], [], [REAC...",Signaling by Receptor Tyrosine Kinases,REAC:R-HSA-9006934,0.000005,[REAC:R-HSA-162582],0.133829,query_1,269,0.069767,True,REAC,516,2399,1,"[FRK, ZBTB4, ESR1, EIF4E2, ETS1, CBX7, ERBB4, ...",1
1,Signal Transduction,11056,41633,104,"[[], [], [REAC], [REAC], [REAC], [], [REAC], [...",Signal Transduction,REAC:R-HSA-162582,0.000005,[REAC:0000000],0.386617,query_1,269,0.040467,True,REAC,2570,2309,1,"[FST, FAM126A, FRK, RORA, ZBTB4, ESR1, NRG1, E...",1
2,Intracellular signaling by second messengers,11056,40641,21,"[[], [], [], [REAC], [REAC], [], [], [], [], [...",Intracellular signaling by second messengers,REAC:R-HSA-9006925,0.003160,[REAC:R-HSA-162582],0.078067,query_1,269,0.070707,True,REAC,297,1317,1,"[FAM126A, FRK, ZBTB4, ESR1, DZIP1, CBX7, SATB2...",1
3,Regulation of KIT signaling,11056,41386,5,"[[], [], [], [], [], [], [], [], [], [], [], [...",Regulation of KIT signaling,REAC:R-HSA-1433559,0.003160,[REAC:R-HSA-1433557],0.018587,query_1,269,0.384615,True,REAC,13,2062,1,"[CBX7, RUNX1T1, ERG, MMP2, ZC2HC1C]",1
4,Signaling by TGFB family members,11056,41731,14,"[[], [], [REAC], [], [], [], [REAC], [], [], [...",Signaling by TGFB family members,REAC:R-HSA-9006936,0.005760,[REAC:R-HSA-162582],0.052045,query_1,269,0.089744,True,REAC,156,2407,1,"[FST, RORA, CALU, CALN1, GBA2, GNB5, DKK3, CDS...",1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1086,Processing of Capped Intron-Containing Pre-mRNA,11056,41193,1,"[[], [], [], [], [], [], [], [], [], [], [], [...",Processing of Capped Intron-Containing Pre-mRNA,REAC:R-HSA-72203,1.000000,[REAC:R-HSA-8953854],0.003717,query_1,269,0.003623,False,REAC,276,1869,1,[KCND2],0
1087,mRNA Splicing,11056,42128,1,"[[], [], [], [], [], [], [], [], [], [], [], [...",mRNA Splicing,REAC:R-HSA-72172,1.000000,[REAC:R-HSA-72203],0.003717,query_1,269,0.004785,False,REAC,209,2804,1,[KCND2],0
1088,mRNA Splicing - Major Pathway,11056,42129,1,"[[], [], [], [], [], [], [], [], [], [], [], [...",mRNA Splicing - Major Pathway,REAC:R-HSA-72163,1.000000,[REAC:R-HSA-72172],0.003717,query_1,269,0.004950,False,REAC,202,2805,1,[KCND2],0
1089,rRNA processing,11056,42153,1,"[[], [], [], [], [], [], [], [], [], [], [], [...",rRNA processing,REAC:R-HSA-72312,1.000000,[REAC:R-HSA-8953854],0.003717,query_1,269,0.004950,False,REAC,202,2829,1,[ZFHX3],0


### Normal

In [13]:
# Perform pathway enrichment for the gene set related to the group
group = 'Normal'
enrichment_df = perform_and_flag_pathway_enrichment(group)

In [14]:
# Print the group's pathway enrichment DataFrame
enrichment_df

,description,effective_domain_size,goshv,intersection_size,intersections,name,native,p_value,parents,precision,query,query_size,recall,significant,source,term_size,source_order,group_id,genes,is_pathway_of_interest
0,Signal Transduction,11056,41633,528,"[[], [], [], [], [REAC], [REAC], [], [], [], [...",Signal Transduction,REAC:R-HSA-162582,2.122938e-15,[REAC:0000000],0.319033,query_1,1655,0.205447,True,REAC,2570,2309,1,"[LRRC1, FZD4, NRP1, HCAR2, GNAI1, ITSN1, RCC2,...",1
1,Membrane Trafficking,11056,40800,154,"[[], [], [], [], [], [REAC], [REAC], [], [], [...",Membrane Trafficking,REAC:R-HSA-199991,5.054395e-08,[REAC:R-HSA-5653656],0.093051,query_1,1655,0.246006,True,REAC,626,1476,1,"[FZD4, RAB5B, ITSN1, YWHAG, DENND2B, GDI2, TPD...",1
2,RHO GTPase cycle,11056,41244,117,"[[], [], [], [], [REAC], [], [], [], [], [], [...",RHO GTPase cycle,REAC:R-HSA-9012999,2.082980e-07,[REAC:R-HSA-194315],0.070695,query_1,1655,0.260579,True,REAC,449,1920,1,"[LRRC1, ITSN1, PAK4, CUL3, TPM3, SOWAHC, ABI2,...",1
3,Signaling by TGFB family members,11056,41731,52,"[[], [], [], [], [], [], [], [], [], [], [], [...",Signaling by TGFB family members,REAC:R-HSA-9006936,3.053422e-06,[REAC:R-HSA-162582],0.031420,query_1,1655,0.333333,True,REAC,156,2407,1,"[CHRDL1, TGIF1, AGO1, ACVR1C, FSTL1, E2F5, PGR...",1
4,Diseases of signal transduction by growth fact...,11056,40135,111,"[[], [], [], [], [], [REAC], [], [], [], [], [...",Diseases of signal transduction by growth fact...,REAC:R-HSA-5663202,8.761302e-06,[REAC:R-HSA-1643685],0.067069,query_1,1655,0.247216,True,REAC,449,811,1,"[FZD4, TP53, SKP1, AGO1, ETV6, TPM3, TRIM24, Q...",1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1959,Activated PKN1 stimulates transcription of AR ...,11056,39398,3,"[[], [], [], [], [], [], [], [], [], [], [], [...",Activated PKN1 stimulates transcription of AR ...,REAC:R-HSA-5625886,1.000000e+00,[REAC:R-HSA-5625740],0.001813,query_1,1655,0.045455,False,REAC,66,74,1,"[NCOA2, H3-3B, SCD]",0
1960,mRNA Capping,11056,42123,1,"[[], [], [], [], [], [], [], [], [], [], [], [...",mRNA Capping,REAC:R-HSA-72086,1.000000e+00,[REAC:R-HSA-8953854],0.000604,query_1,1655,0.035714,False,REAC,28,2799,1,[THAP11],0
1961,Abortive elongation of HIV-1 transcript in the...,11056,39379,1,"[[], [], [], [], [], [], [], [], [], [], [], [...",Abortive elongation of HIV-1 transcript in the...,REAC:R-HSA-167242,1.000000e+00,[REAC:R-HSA-167169],0.000604,query_1,1655,0.045455,False,REAC,22,55,1,[THAP11],0
1962,REACTOME root term,11056,42173,1655,"[[REAC], [REAC], [], [REAC], [REAC], [REAC], [...",REACTOME root term,REAC:0000000,1.000000e+00,[],1.000000,query_1,1655,0.149692,False,REAC,11056,2849,1,"[RPS3, RPA1, RPL14, LRRC1, FZD4, RAB5B, KAT7, ...",0


# Layered Tripartite Networks
MicroRNA -- Messenger RNA -- Pathway

## Functions

### Build Membership Network

In [15]:

def build_membership_network(group_dir):
    """
    Build a bipartite gene-pathway membership network from enrichment results.

    This function loads the functional enrichment results for a given group,
    filters the pathways of interest, and constructs a bipartite network
    representing gene-pathway membership relationships. The network is
    represented by two DataFrames containing edges and nodes, which are
    stored in processed CSV files. The file paths and filenames are
    obtained from global configuration dictionaries.

    Parameters
    ----------
    group_dir : str
        The normalized name of the group for which the membership network
        will be constructed. Must be a key in
        `NETWORK_DATA_DIRS['processed']`.

    Returns
    -------
    membership_edges_df : pandas.DataFrame
        DataFrame containing the gene-pathway membership edges, with
        the following columns:
        - `source` : Gene/messenger RNA identifier.
        - `target` : Pathway identifier.

    membership_nodes_df : pandas.DataFrame
        DataFrame containing the gene/messenger RNA and pathway nodes,
        with the following columns:
        - `id` : Unique identifier of the node.
        - `label` : Gene identifier or pathway name.
        - `type` : Node type (`Messenger RNA` or `Pathway`).
        - `qvalue` : Adjusted p-value of the pathway enrichment result,
          available only for pathway nodes.

    Notes
    -----
    - The following global configuration variables are used:
        * `GP_DATA_DIRS` : Directory paths for functional enrichment data.
        * `GP_FILE_SUFFIX` : Filename suffix for enrichment results.
        * `NETWORK_DATA_DIRS` : Directory paths for processed network data.
        * `NETWORK_FILES` : Filenames for specific network data files.
    - Only enrichment results flagged with `is_pathway_of_interest == 1`
      are considered for network construction.
    - The `genes` column is converted from string representations into
      Python lists before being expanded into individual membership edges.
    - Each edge represents the membership of a gene/messenger RNA in an
      enriched pathway.
    - Duplicate gene/messenger RNA nodes are removed to ensure that each
      gene is represented only once in the node DataFrame.
    - The function overwrites the corresponding edge and node CSV files
      in the processed directory.

    Examples
    --------
    >>> edges_df, nodes_df = build_membership_network('her2-enriched')
    >>> edges_df.head()
       source       target
    0   BRCA1  R-HSA-12345
    1    TP53  R-HSA-12345
    2    EGFR  R-HSA-67890

    >>> nodes_df[['id', 'label', 'type']].head()
              id         label          type
    0      BRCA1         BRCA1  Messenger RNA
    1       TP53          TP53  Messenger RNA
    2       EGFR          EGFR  Messenger RNA
    3 R-HSA-12345     Pathway A       Pathway
    4 R-HSA-67890     Pathway B       Pathway
    """
    # Define the path to the directory
    dir_path = GP_DATA_DIRS['processed']

    # Create a DataFrame of pathway enrichment results
    file_name = group_dir + GP_FILE_SUFFIX
    enrichment_df = pd.read_csv(os.path.join(dir_path, file_name))

    # Filter columns and results of interest
    cols = [
        'native',
        'name',
        'p_value',
        'genes',
        'is_pathway_of_interest'
    ]
    enrichment_df = (
        enrichment_df[cols]
        .query('is_pathway_of_interest == 1')
        .drop(columns='is_pathway_of_interest')
        .reset_index(drop=True)
    )

    # Create a DataFrame of pathway nodes
    pathways_df = (
        enrichment_df
        .rename(columns={
            'native': 'id',
            'name': 'label',
            'p_value': 'qvalue',
        })
        [['id', 'label', 'qvalue']]
    )
    pathways_df['type'] = 'Pathway'

    # Create a DataFrame of edges between genes and pathways
    membership_edges_df = (
        enrichment_df
        .assign(genes=lambda df: df['genes'].apply(ast.literal_eval))
        .explode(column='genes', ignore_index=True)
        .dropna(subset=['genes'])
        .rename(columns={
            'genes': 'source',
            'native': 'target',
        })
        [['source', 'target']]
    )

    # Create a DataFrame of enriched gene/messenger RNA nodes
    enriched_genes_df = (
        membership_edges_df[['source']]
        .rename(columns={'source': 'id'})
        .drop_duplicates(ignore_index=True)
    )
    enriched_genes_df['label'] = enriched_genes_df['id']
    enriched_genes_df['type'] = 'Messenger RNA'

    # Concatenate enriched gene and pathway nodes in a DataFrame
    membership_nodes_df = pd.concat(
        objs=[enriched_genes_df, pathways_df],
        ignore_index=True
    )

    # Store the DataFrame of edges
    file_name = NETWORK_FILES['membership-edges']
    dir_path = NETWORK_DATA_DIRS['processed'][group_dir]
    membership_edges_df.to_csv(
        os.path.join(dir_path, file_name), index=False
    )

    # Store the DataFrame of nodes
    file_name = NETWORK_FILES['membership-nodes']
    membership_nodes_df.to_csv(
        os.path.join(dir_path, file_name), index=False
    )

    return membership_edges_df, membership_nodes_df

### Build Tripartite Network

In [16]:

def build_tripartite_network(group):
    """
    Build a layered tripartite microRNA-mRNA-pathway network for a given group.

    This function loads the nodes and edges of a microRNA-mRNA interaction
    network, constructs the corresponding mRNA-pathway membership network,
    and integrates both networks into a tripartite representation. The
    resulting network is represented by two DataFrames containing edges
    and nodes, which are stored in processed CSV files. The file paths
    and filenames are obtained from global configuration dictionaries.

    Parameters
    ----------
    group : str
        The name of the group for which the tripartite network will be
        constructed. The normalized group name must correspond to a key
        in `NETWORK_DATA_DIRS['processed']`.

    Returns
    -------
    tripartite_edges_df : pandas.DataFrame
        DataFrame containing the tripartite network edges, representing
        microRNA-mRNA interactions and mRNA-pathway memberships.

    tripartite_nodes_df : pandas.DataFrame
        DataFrame containing the tripartite network nodes, including
        microRNAs, messenger RNAs, and pathways, with their associated
        attributes.

    Notes
    -----
    - The following global configuration variables are used:
        * `NETWORK_DATA_DIRS` : Directory paths for processed network data.
        * `NETWORK_FILES` : Filenames for specific network data files.
    - The group name is normalized by converting it to lowercase and
      replacing spaces with hyphens.
    - The function uses `build_membership_network()` to construct the
      mRNA-pathway membership network from functional enrichment results.
    - The tripartite network integrates two types of relationships:
        * MicroRNA-mRNA interactions inferred from expression data.
        * mRNA-pathway memberships identified through enrichment analysis.
    - Duplicate node rows are removed from the integrated node DataFrame.
    - The function overwrites the corresponding edge and node CSV files
      in the processed directory.

    Examples
    --------
    >>> edges_df, nodes_df = build_tripartite_network('HER2-enriched')
    >>> edges_df[['source', 'target']].head()
          source       target
    0     miR-21        BRCA1
    1    miR-155         TP53
    2      BRCA1  R-HSA-12345
    3       TP53  R-HSA-67890

    >>> nodes_df[['id', 'label', 'type']].head()
              id         label          type
    0     miR-21        miR-21       MicroRNA
    1    miR-155       miR-155       MicroRNA
    2      BRCA1         BRCA1  Messenger RNA
    3       TP53          TP53  Messenger RNA
    4 R-HSA-12345     Pathway A       Pathway
    """
    # Define the path to the directory related to the group
    group_dir = (group.lower()).replace(' ', '-')
    dir_path = NETWORK_DATA_DIRS['processed'][group_dir]

    # Create a DataFrame for the edges of the miR-mRNA interaction network
    file_name = NETWORK_FILES['interaction-edges']
    interaction_edges_df = pd.read_csv(os.path.join(dir_path, file_name))

    # Create a DataFrame for the nodes of the miR-mRNA interaction network
    file_name = NETWORK_FILES['interaction-nodes']
    interaction_nodes_df = pd.read_csv(os.path.join(dir_path, file_name))

    # Build the mRNA-pathway membership network
    membership_edges_df, membership_nodes_df = \
        build_membership_network(group_dir)

    # Concatenate the edges in a DataFrame
    tripartite_edges_df = pd.concat(
        objs=[interaction_edges_df, membership_edges_df],
        ignore_index=True
    )

    # Concatenate the nodes in a DataFrame
    tripartite_nodes_df = (
        pd.concat(objs=[interaction_nodes_df, membership_nodes_df])
        .drop_duplicates(subset='id', ignore_index=True)
    )

    # Store the DataFrame of edges
    file_name = NETWORK_FILES['tripartite-edges']
    tripartite_edges_df.to_csv(
        os.path.join(dir_path, file_name), index=False
    )

    # Store the DataFrame of nodes
    file_name = NETWORK_FILES['tripartite-nodes']
    tripartite_nodes_df.to_csv(
        os.path.join(dir_path, file_name), index=False
    )

    return tripartite_edges_df, tripartite_nodes_df

## Groups

### Basal-like

In [17]:
# Build the group's tripartite network
group = 'Basal-like'
edges_df, nodes_df = build_tripartite_network(group)

In [18]:
# Print the group's edges DataFrame
edges_df

,source,target,mirtarbase,correlation,qvalue
0,hsa-miR-182-5p,RAB27A,NaN,-0.564,0.000
1,hsa-miR-18a-5p,ORAI3,MIRT050737,-0.545,0.000
2,hsa-miR-18a-5p,ZBTB4,NaN,-0.532,0.001
3,hsa-miR-96-5p,RAB27A,NaN,-0.496,0.004
4,hsa-miR-106b-5p,ATXN1,MIRT441140,-0.495,0.004
...,...,...,...,...,...
301,RXRG,REAC:R-HSA-383280,NaN,NaN,NaN
302,TP53INP2,REAC:R-HSA-383280,NaN,NaN,NaN
303,ARHGEF3,REAC:R-HSA-383280,NaN,NaN,NaN
304,SMARCD1,REAC:R-HSA-383280,NaN,NaN,NaN


In [19]:
# Print the group's nodes DataFrame
nodes_df

,id,label,type,qvalue
0,hsa-miR-182-5p,hsa-miR-182-5p,MicroRNA,NaN
1,hsa-miR-18a-5p,hsa-miR-18a-5p,MicroRNA,NaN
2,hsa-miR-96-5p,hsa-miR-96-5p,MicroRNA,NaN
3,hsa-miR-106b-5p,hsa-miR-106b-5p,MicroRNA,NaN
4,hsa-miR-221-3p,hsa-miR-221-3p,MicroRNA,NaN
...,...,...,...,...
288,DYNLL2,DYNLL2,Messenger RNA,NaN
289,TGFBR2,TGFBR2,Messenger RNA,NaN
290,DHX36,DHX36,Messenger RNA,NaN
291,REAC:R-HSA-5576892,Phase 0 - rapid depolarisation,Pathway,0.038079


### HER2-enriched

In [20]:
# Build the group's tripartite network
group = 'HER2-enriched'
edges_df, nodes_df = build_tripartite_network(group)

In [21]:
# Print the group's edges DataFrame
edges_df

,source,target,mirtarbase,correlation,qvalue
0,hsa-miR-141-3p,IGF2,NaN,-0.587,0.032
1,hsa-let-7d-5p,MEF2D,MIRT735654,-0.562,0.032
2,hsa-miR-378a-3p,ACP6,NaN,-0.562,0.032
3,hsa-miR-18a-5p,ZBTB20,NaN,-0.559,0.032
4,hsa-miR-106b-5p,PLXDC2,NaN,-0.558,0.032
5,hsa-let-7d-5p,SMARCC1,NaN,-0.546,0.042
6,hsa-miR-106b-5p,SALL1,NaN,-0.545,0.042
7,hsa-let-7d-5p,HIF1AN,NaN,-0.539,0.042
8,hsa-let-7d-5p,PRKAR2A,NaN,-0.537,0.042
9,hsa-miR-155-5p,VAV3,NaN,-0.537,0.042


In [22]:
# Print the group's nodes DataFrame
nodes_df

,id,label,type,qvalue
0,hsa-miR-141-3p,hsa-miR-141-3p,MicroRNA,NaN
1,hsa-let-7d-5p,hsa-let-7d-5p,MicroRNA,NaN
2,hsa-miR-378a-3p,hsa-miR-378a-3p,MicroRNA,NaN
3,hsa-miR-18a-5p,hsa-miR-18a-5p,MicroRNA,NaN
4,hsa-miR-106b-5p,hsa-miR-106b-5p,MicroRNA,NaN
5,hsa-miR-155-5p,hsa-miR-155-5p,MicroRNA,NaN
6,hsa-miR-182-5p,hsa-miR-182-5p,MicroRNA,NaN
7,hsa-miR-320b,hsa-miR-320b,MicroRNA,NaN
8,hsa-miR-30c-5p,hsa-miR-30c-5p,MicroRNA,NaN
9,IGF2,IGF2,Messenger RNA,NaN


### Luminal A

In [23]:
# Build the group's tripartite network
group = 'Luminal A'
edges_df, nodes_df = build_tripartite_network(group)

In [24]:
# Print the group's edges DataFrame
edges_df

,source,target,mirtarbase,correlation,qvalue
0,hsa-miR-200a-3p,WWTR1,NaN,-0.477,0.0
1,hsa-miR-141-3p,PHYHIPL,NaN,-0.452,0.0
2,hsa-miR-96-5p,SGMS2,NaN,-0.438,0.0
3,hsa-miR-29b-3p,PCSK5,NaN,-0.425,0.0
4,hsa-miR-342-3p,FRMD4A,NaN,-0.425,0.0
...,...,...,...,...,...
494,C2CD2,REAC:R-HSA-9675108,NaN,NaN,NaN
495,ATP11C,REAC:R-HSA-9675108,NaN,NaN,NaN
496,CROT,REAC:R-HSA-9675108,NaN,NaN,NaN
497,IGDCC4,REAC:R-HSA-9675108,NaN,NaN,NaN


In [25]:
# Print the group's nodes DataFrame
nodes_df

,id,label,type,qvalue
0,hsa-miR-200a-3p,hsa-miR-200a-3p,MicroRNA,NaN
1,hsa-miR-141-3p,hsa-miR-141-3p,MicroRNA,NaN
2,hsa-miR-96-5p,hsa-miR-96-5p,MicroRNA,NaN
3,hsa-miR-29b-3p,hsa-miR-29b-3p,MicroRNA,NaN
4,hsa-miR-342-3p,hsa-miR-342-3p,MicroRNA,NaN
...,...,...,...,...
274,REAC:R-HSA-9762293,Regulation of CDH11 gene transcription,Pathway,0.096172
275,REAC:R-HSA-5663202,Diseases of signal transduction by growth fact...,Pathway,0.096172
276,REAC:R-HSA-170834,Signaling by TGF-beta Receptor Complex,Pathway,0.096172
277,REAC:R-HSA-2219528,PI3K/AKT Signaling in Cancer,Pathway,0.096172


### Luminal B

In [26]:
# Build the group's tripartite network
group = 'Luminal B'
edges_df, nodes_df = build_tripartite_network(group)

In [27]:
# Print the group's edges DataFrame
edges_df

,source,target,mirtarbase,correlation,qvalue
0,hsa-miR-200c-3p,ZMAT3,MIRT209182,-0.493,0.000
1,hsa-let-7g-5p,KLHL13,NaN,-0.467,0.001
2,hsa-miR-30d-5p,FST,NaN,-0.461,0.001
3,hsa-miR-200c-3p,FAM126A,NaN,-0.458,0.001
4,hsa-miR-151a-3p,FRK,MIRT508167,-0.456,0.001
...,...,...,...,...,...
1057,KCTD20,REAC:R-HSA-5663202,NaN,NaN,NaN
1058,KIRREL1,REAC:R-HSA-5663202,NaN,NaN,NaN
1059,ZC2HC1C,REAC:R-HSA-5663202,NaN,NaN,NaN
1060,AFF3,REAC:R-HSA-5663202,NaN,NaN,NaN


In [28]:
# Print the group's nodes DataFrame
nodes_df

,id,label,type,qvalue
0,hsa-miR-200c-3p,hsa-miR-200c-3p,MicroRNA,NaN
1,hsa-let-7g-5p,hsa-let-7g-5p,MicroRNA,NaN
2,hsa-miR-30d-5p,hsa-miR-30d-5p,MicroRNA,NaN
3,hsa-miR-151a-3p,hsa-miR-151a-3p,MicroRNA,NaN
4,hsa-miR-27b-3p,hsa-miR-27b-3p,MicroRNA,NaN
...,...,...,...,...
593,REAC:R-HSA-4086398,Ca2+ pathway,Pathway,0.093552
594,REAC:R-HSA-194306,Neurophilin interactions with VEGF and VEGFR,Pathway,0.095842
595,REAC:R-HSA-8941237,Invadopodia formation,Pathway,0.095842
596,REAC:R-HSA-211163,AKT-mediated inactivation of FOXO1A,Pathway,0.095842


### Normal

In [29]:
# Build the group's tripartite network
group = 'Normal'
edges_df, nodes_df = build_tripartite_network(group)

In [30]:
# Print the group's edges DataFrame
edges_df

,source,target,mirtarbase,correlation,qvalue
0,hsa-miR-22-3p,RPS3,NaN,-0.816,0.0
1,hsa-miR-452-5p,RPA1,NaN,-0.810,0.0
2,hsa-miR-141-3p,TRHDE,NaN,-0.807,0.0
3,hsa-miR-22-3p,RPL14,NaN,-0.804,0.0
4,hsa-miR-22-3p,LRRC1,MIRT030660,-0.800,0.0
...,...,...,...,...,...
12124,VGLL4,REAC:R-HSA-168799,NaN,NaN,NaN
12125,TBC1D30,REAC:R-HSA-168799,NaN,NaN,NaN
12126,ARL5A,REAC:R-HSA-168799,NaN,NaN,NaN
12127,HIF1AN,REAC:R-HSA-168799,NaN,NaN,NaN


In [31]:
# Print the group's nodes DataFrame
nodes_df

,id,label,type,qvalue
0,hsa-miR-22-3p,hsa-miR-22-3p,MicroRNA,NaN
1,hsa-miR-452-5p,hsa-miR-452-5p,MicroRNA,NaN
2,hsa-miR-141-3p,hsa-miR-141-3p,MicroRNA,NaN
3,hsa-miR-200a-3p,hsa-miR-200a-3p,MicroRNA,NaN
4,hsa-miR-224-5p,hsa-miR-224-5p,MicroRNA,NaN
...,...,...,...,...
3111,REAC:R-HSA-8853884,Transcriptional Regulation by VENTX,Pathway,0.093989
3112,REAC:R-HSA-74752,Signaling by Insulin receptor,Pathway,0.094678
3113,REAC:R-HSA-425381,Bicarbonate transporters,Pathway,0.095304
3114,REAC:R-HSA-450341,Activation of the AP-1 family of transcription...,Pathway,0.095304
